In [ ]:
# ====================== 导入所需库 ======================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 绘图中文设置
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# ====================== 1. 定义文件根路径 & 读取全部数据表 ======================
base_path = "D:/olist/"

customers = pd.read_csv(base_path + "olist_customers_dataset.csv")
order_items = pd.read_csv(base_path + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(base_path + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(base_path + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(base_path + "olist_orders_dataset.csv")
products = pd.read_csv(base_path + "olist_products_dataset.csv")
sellers = pd.read_csv(base_path + "olist_sellers_dataset.csv")
category_trans = pd.read_csv(base_path + "product_category_name_translation.csv")
# geolocation数据量巨大，暂时不并入主宽表，后续地理可视化单独使用
geolocation = pd.read_csv(base_path + "olist_geolocation_dataset.csv")

print("数据表读取完成！")

In [ ]:
# ====================== 2. 缺失值探查函数 ======================
def check_missing(df, table_name):
    miss_count = df.isnull().sum()
    miss_ratio = (df.isnull().sum() / len(df)).round(4)
    miss_df = pd.DataFrame({"缺失数量": miss_count, "缺失率": miss_ratio})
    miss_df = miss_df[miss_df["缺失数量"] > 0]
    print(f"\n==== {table_name} 缺失值统计 ====")
    print(miss_df)

# 查看核心原始表缺失情况
check_missing(orders, "orders订单主表")
check_missing(products, "products商品表")
check_missing(order_reviews, "order_reviews评价表")

In [ ]:
# ====================== 3. 时间字段统一转为datetime格式 ======================
# orders表时间转换
time_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]
for col in time_cols_orders:
    orders[col] = pd.to_datetime(orders[col])

# 订单商品明细表时间
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"])

In [ ]:
# ====================== 4. 多表左连接，构建原始主宽表 df_main ======================
# 关联逻辑：orders订单主表为中心
df_main = pd.merge(orders, customers, on="customer_id", how="left")
df_main = pd.merge(df_main, order_items, on="order_id", how="left")
df_main = pd.merge(df_main, products, on="product_id", how="left")
df_main = pd.merge(df_main, category_trans, on="product_category_name", how="left")
df_main = pd.merge(df_main, order_reviews, on="order_id", how="left")

print(f"\n多表合并完成，原始宽表样本量：{df_main.shape[0]}")

In [ ]:
# ====================== 5. 衍生业务特征（物流、标签、时间特征） ======================
# 实际签收总时长（小时）
df_main["delivery_hours"] = (df_main["order_delivered_customer_date"] - df_main["order_purchase_timestamp"]).dt.total_seconds() / 3600
# 是否物流延迟：实际签收 > 预估签收
df_main["is_delay"] = (df_main["order_delivered_customer_date"] > df_main["order_estimated_delivery_date"]).astype(int)
# 差评标签：评分≤3分为差评 1，4/5分为好评 0
df_main["is_bad_review"] = np.where(df_main["review_score"] <= 3, 1, 0)
# 提取时间特征
df_main["order_year"] = df_main["order_purchase_timestamp"].dt.year
df_main["order_month"] = df_main["order_purchase_timestamp"].dt.month
df_main["order_weekday"] = df_main["order_purchase_timestamp"].dt.weekday

In [ ]:
# ====================== 6. 分层数据清洗（核心！应用统计缺失处理策略） ======================
print(f"\n清洗前样本量：{df_main.shape[0]}")

# 6.1 筛选：只保留【成功交付订单】，剔除取消、未发货订单（时间字段MAR非随机缺失）
df_main_clean = df_main[df_main["order_delivered_customer_date"].notna()].copy()

# 6.2 商品重量：同品类分组均值填充
df_main_clean["product_weight_g"] = df_main_clean.groupby("product_category_name")["product_weight_g"].transform(
    lambda x: x.fillna(x.mean())
)

# 6.3 剔除商品品类为空的样本（后续品类分析、建模需要）
df_main_clean = df_main_clean[df_main_clean["product_category_name"].notna()].copy()

print(f"清洗后有效样本量：{df_main_clean.shape[0]}")
check_missing(df_main_clean, "清洗完成后的最终宽表")

In [ ]:
# ====================== 7. 导出清洗完毕宽表 ======================
df_main_clean.to_csv("D:/olist/df_main_wide.csv", index=False, encoding="utf-8-sig")
print("\n✅ 清洗后的宽表已保存至 D:/olist/df_main_wide.csv")

In [ ]:
# ====================== 8. 基础描述统计（偏度、峰度，写入项目报告） ======================
num_analysis_cols = ["delivery_hours", "review_score"]
print("\n==== 关键数值变量描述统计（均值、中位数、偏度、峰度） ====")
for col in num_analysis_cols:
    data = df_main_clean[col].dropna()
    print(f"\n【{col}】")
    print(f"均值 = {data.mean():.2f}，中位数 = {data.median():.2f}")
    print(f"偏度 = {stats.skew(data):.3f}，峰度 = {stats.kurtosis(data):.3f}")